In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pickle

In [19]:
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc.p")
print(type(data))
print(data)


<class 'dict'>
{'obs':          cid         lon        lat                time          z         SA  \
0        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.148984   
1        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149086   
2        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149387   
3        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149889   
4        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -5.255167  31.145790   
...      ...         ...        ...                 ...        ...        ...   
5013  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -24.400000        NaN   
5014  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -33.900000        NaN   
5015  3351.0 -122.428001  47.744000 2014-12-15 17:42:00 -14.900000        NaN   
5016  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -0.620000        NaN   
5017  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -1.600000        NaN   

    

In [ ]:
# resave as .pkl file to be able to open with loenv

with open("combined_bottle_2014_cas7_t1_x11ab_ssc.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc.pkl")
print(type(data))
print(data)

In [8]:
print(list(data.keys()))

['obs', 'cas7_t1_x11ab', 'ssc', 'meta']


In [9]:
print(data["obs"].columns)
print(data["cas7_t1_x11ab"].columns)
print(data["ssc"].columns)

Index(['cid', 'lon', 'lat', 'time', 'z', 'SA', 'CT', 'DO', 'NO3', 'Chl',
       'name', 'cruise', 'source', 'NH4', 'PO4 (uM)', 'SiO4 (uM)', 'NO2 (uM)',
       'TA', 'DIC'],
      dtype='object')
Index(['time', 'lat', 'lon', 'z', 'cid', 'cruise', 'name', 'source', 'CT',
       'SA', 'DO', 'Chl', 'NO3', 'NH4', 'TA', 'DIC'],
      dtype='object')
Index(['time', 'lat', 'lon', 'z', 'NO3', 'silicon', 'NH4', 'DIAT', 'FLAG',
       'SA', 'CT', 'TA', 'DIC', 'DO'],
      dtype='object')


In [10]:
# assign ssc name and cid based on lat/lon and time

# make lookup table
stn_df = data['cas7_t1_x11ab'].groupby('name', as_index=False).first()
stn_df = stn_df[['name', 'lat','lon']].copy()

ssc = data['ssc'].copy()

# round coordinates to 2 decimal places (~1 km) to allow for small mismatches
ndec = 2
stn_df['lat_r'] = stn_df['lat'].round(ndec)
stn_df['lon_r'] = stn_df['lon'].round(ndec)

ssc['lat_r'] = ssc['lat'].round(ndec)
ssc['lon_r'] = ssc['lon'].round(ndec)

ssc = ssc.merge(
    stn_df[['name', 'lat_r', 'lon_r']],
    on=['lat_r', 'lon_r'],
    how='left'
)

# check how many failed to match
n_bad = ssc['name'].isna().sum()
print(f"SSC rows with no exact lat/lon match: {n_bad} out of {len(ssc)}")

# drop helper columns if you want
ssc = ssc.drop(columns=['lat_r', 'lon_r'])

# save back
data['ssc'] = ssc

SSC rows with no exact lat/lon match: 1725 out of 5274


In [11]:
# check method on cas7_t1_x11ab as well

# make lookup table
stn_df = data['obs'].groupby('name', as_index=False).first()
stn_df = stn_df[['name', 'lat','lon']].copy()

cas7_t1_x11ab = data['cas7_t1_x11ab'].copy()
cas7_t1_x11ab = cas7_t1_x11ab.drop(columns=['name'])

# round coordinates to 2 decimal places
ndec = 2
stn_df['lat_r'] = stn_df['lat'].round(ndec)
stn_df['lon_r'] = stn_df['lon'].round(ndec)

cas7_t1_x11ab['lat_r'] = cas7_t1_x11ab['lat'].round(ndec)
cas7_t1_x11ab['lon_r'] = cas7_t1_x11ab['lon'].round(ndec)

cas7_t1_x11ab = cas7_t1_x11ab.merge(
    stn_df[['name', 'lat_r', 'lon_r']],
    on=['lat_r', 'lon_r'],
    how='left'
)

# check how many failed to match
n_bad = cas7_t1_x11ab['name'].isna().sum()
print(f"cas7 rows with no exact lat/lon match: {n_bad} out of {len(cas7_t1_x11ab)}")

# drop helper columns if you want
cas7_t1_x11ab = cas7_t1_x11ab.drop(columns=['lat_r', 'lon_r'])

cas7 rows with no exact lat/lon match: 1725 out of 5274


In [12]:
cas7_t1_x11ab = data['cas7_t1_x11ab'].copy()
n_bad = cas7_t1_x11ab['name'].isna().sum()
print(f"rows with no exact lat/lon match: {n_bad} out of {len(cas7_t1_x11ab)}")

rows with no exact lat/lon match: 1606 out of 5018


In [13]:
# ensure datetime
data['obs']['time'] = pd.to_datetime(data['obs']['time'])
ssc = data['ssc'].copy()
ssc['time'] = pd.to_datetime(ssc['time'])

# make obs lookup table: (name, time) -> cid
obs_lookup = data['obs'].groupby('cid', as_index=False).first()
obs_lookup = obs_lookup[['cid', 'name', 'time']]

# merge SSC with obs cid
ssc = ssc.merge(
    obs_lookup,
    on=['name', 'time'],
    how='left'
)

# check failures
n_bad = ssc['cid'].isna().sum()
print(f"SSC rows with no cid match: {n_bad} out of {len(ssc)}")

# save back
data['ssc'] = ssc

SSC rows with no cid match: 1345 out of 5274


In [14]:
# check method on cas7_t1_x11ab as well
# ensure datetime
data['obs']['time'] = pd.to_datetime(data['obs']['time'])
cas7_t1_x11ab = data['cas7_t1_x11ab'].copy()
cas7_t1_x11ab['time'] = pd.to_datetime(cas7_t1_x11ab['time'])

# make obs lookup table: (name, time) -> cid
obs_lookup = data['obs'].groupby('cid', as_index=False).first()
obs_lookup = obs_lookup[['cid', 'name', 'time']]

# merge SSC with obs cid
ssc = ssc.merge(
    obs_lookup,
    on=['name', 'time'],
    how='left'
)

# check failures
n_bad = cas7_t1_x11ab['cid'].isna().sum()
print(f"rows with no cid match: {n_bad} out of {len(cas7_t1_x11ab)}")

rows with no cid match: 0 out of 5018


In [15]:
print(data["obs"].columns)
print(data["cas7_t1_x11ab"].columns)
print(data["ssc"].columns)

Index(['cid', 'lon', 'lat', 'time', 'z', 'SA', 'CT', 'DO', 'NO3', 'Chl',
       'name', 'cruise', 'source', 'NH4', 'PO4 (uM)', 'SiO4 (uM)', 'NO2 (uM)',
       'TA', 'DIC'],
      dtype='object')
Index(['time', 'lat', 'lon', 'z', 'cid', 'cruise', 'name', 'source', 'CT',
       'SA', 'DO', 'Chl', 'NO3', 'NH4', 'TA', 'DIC'],
      dtype='object')
Index(['time', 'lat', 'lon', 'z', 'NO3', 'silicon', 'NH4', 'DIAT', 'FLAG',
       'SA', 'CT', 'TA', 'DIC', 'DO', 'name', 'cid'],
      dtype='object')


In [11]:
out_fn = "./combined_bottle_2014_cas7_t1_x11ab_ssc.pkl"
pd.to_pickle(data, out_fn)